In [1]:
# !pip install openai
# !pip install python-dotenv

In [1]:
import os
import openai
import sys
sys.path.append('./')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.environ['OPENAI_API_KEY']

## 1단계: 어시스턴트 생성
어시스턴트는 다음과 같은 여러 매개변수를 사용하여 사용자의 메시지에 응답하도록 구성할 수 있는 엔터티를 나타냅니다.  

Instruction: 어시스턴트와 모델이 어떻게 행동하거나 반응해야 하는지 지시  
Model: GPT-3.5 또는 GPT-4 모델을 지정할 수 있습니다. 검색 도구에는 최소한 gpt-3.5-turbo-1106(최신 버전 지원) 또는 gpt-4-turbo-preview 모델이 필요.  
Tools: API는 OpenAI에서 구축하고 호스팅하는 코드 해석기 및 검색을 지원합니다.  
Functions: API를 사용하면 함수 호출 기능과 유사한 동작으로 사용자 정의 함수 서명을 정의할 수 있습니다.  
이 예에서는 코드 해석기 도구가 활성화된 개인 수학 교사인 도우미를 만듭니다.  

In [3]:
from openai import OpenAI
client = OpenAI()

In [4]:
assistant = client.beta.assistants.create(
    name="Math Tutor",
    instructions="You are a personal math tutor. Write and run code to answer math questions.",
    tools=[{"type": "code_interpreter"}],
    model="gpt-4-turbo-preview"
)

In [18]:
vars(assistant)

{'id': 'asst_SFlnTtCdxB0TFI8f88wBVpdj',
 'created_at': 1709722809,
 'description': None,
 'file_ids': [],
 'instructions': 'You are a personal math tutor. Write and run code to answer math questions.',
 'metadata': {},
 'model': 'gpt-4-turbo-preview',
 'name': 'Math Tutor',
 'object': 'assistant',
 'tools': [ToolCodeInterpreter(type='code_interpreter')]}

## 2단계: 스레드 생성
스레드는 대화를 나타냅니다. 사용자가 대화를 시작하자마자 사용자당 하나의 스레드를 생성하는 것이 좋습니다. 메시지를 생성하여 이 스레드에 사용자별 컨텍스트와 파일을 전달합니다  
스레드에는 크기 제한이 없습니다. 스레드에 원하는 만큼 메시지를 추가할 수 있습니다. Assistant는 ChatGPT로 광범위하게 테스트한 잘림과 같은 관련 최적화 기술을 사용하여 모델에 대한 요청이 최대 컨텍스트 창에 맞는지 확인합니다. 어시스턴트 API를 사용하면 특정 실행에 대해 모델에 전달되는 입력 토큰 수에 대한 제어를 위임합니다. 즉, 경우에 따라 어시스턴트 실행 비용에 대한 제어 권한이 줄어들지만 복잡성을 처리할 필요는 없습니다. 컨텍스트 창을 직접 관리하는 것입니다..

In [5]:
thread = client.beta.threads.create()
thread

Thread(id='thread_Wc1mZUQj7mnDFqgfp3UUenai', created_at=1709722929, metadata={}, object='thread')

## 3단계: 스레드에 메시지 추가
메시지에는 텍스트와 선택적으로 사용자가 업로드할 수 있는 파일이 포함됩니다. 특정 스레드에 메시지를 추가해야 합니다..

In [6]:
message = client.beta.threads.messages.create(
    thread_id=thread.id,
    role="user",
    content="I need to solve the equation `3x + 11 = 14`. Can you help me?"
)
message

ThreadMessage(id='msg_5IdwxEG2C8RiJJ9wqQDaIQIr', assistant_id=None, content=[MessageContentText(text=Text(annotations=[], value='I need to solve the equation `3x + 11 = 14`. Can you help me?'), type='text')], created_at=1709722981, file_ids=[], metadata={}, object='thread.message', role='user', run_id=None, thread_id='thread_Wc1mZUQj7mnDFqgfp3UUenai')

In [12]:
vars(message)

{'id': 'msg_5IdwxEG2C8RiJJ9wqQDaIQIr',
 'assistant_id': None,
 'content': [MessageContentText(text=Text(annotations=[], value='I need to solve the equation `3x + 11 = 14`. Can you help me?'), type='text')],
 'created_at': 1709722981,
 'file_ids': [],
 'metadata': {},
 'object': 'thread.message',
 'role': 'user',
 'run_id': None,
 'thread_id': 'thread_Wc1mZUQj7mnDFqgfp3UUenai'}

## 4단계: Assistant실행
Assistant가 사용자 메시지에 응답하려면 Run을 생성해야 합니다. 이를 통해 Assistant는 스레드를 읽고 도구를 호출할지(활성화된 경우) 아니면 단순히 모델을 사용하여 쿼리에 가장 잘 응답할지 여부를 결정합니다. 실행이 진행됨에 따라assistant자이role= "assistant"인 스레드에 메시지를 추가합니다. 또한 Assistant는 모델의 컨텍스트 창에 포함할 이전 메시지를 자동으로 결정합니다. 이는 가격뿐만 아니라 모델 성능에도 영향을 미칩니다니다.

In [13]:
run = client.beta.threads.runs.create(
  thread_id=thread.id,
  assistant_id=assistant.id,
  instructions="Please address the user as Jane Doe. The user has a premium account."
)

In [16]:
vars(run)

{'id': 'run_ENnFp00ccuQqykYOl3GPgNaY',
 'assistant_id': 'asst_SFlnTtCdxB0TFI8f88wBVpdj',
 'cancelled_at': None,
 'completed_at': None,
 'created_at': 1709723528,
 'expires_at': 1709724128,
 'failed_at': None,
 'file_ids': [],
 'instructions': 'Please address the user as Jane Doe. The user has a premium account.',
 'last_error': None,
 'metadata': {},
 'model': 'gpt-4-turbo-preview',
 'object': 'thread.run',
 'required_action': None,
 'started_at': None,
 'status': 'queued',
 'thread_id': 'thread_Wc1mZUQj7mnDFqgfp3UUenai',
 'tools': [ToolAssistantToolsCode(type='code_interpreter')],
 'usage': None}

## 5단계: 실행 상태 확인
기본적으로 실행은 대기열 상태로 전환됩니다. 정기적으로 실행을 검색하여 상태를 확인하고 완료됨으로 이동했는지 확인할 수 있습니다.

In [21]:
run = client.beta.threads.runs.retrieve(
  thread_id=thread.id,
  run_id=run.id
)
run

Run(id='run_ENnFp00ccuQqykYOl3GPgNaY', assistant_id='asst_SFlnTtCdxB0TFI8f88wBVpdj', cancelled_at=None, completed_at=1709723546, created_at=1709723528, expires_at=None, failed_at=None, file_ids=[], instructions='Please address the user as Jane Doe. The user has a premium account.', last_error=None, metadata={}, model='gpt-4-turbo-preview', object='thread.run', required_action=None, started_at=1709723528, status='completed', thread_id='thread_Wc1mZUQj7mnDFqgfp3UUenai', tools=[ToolAssistantToolsCode(type='code_interpreter')], usage={'prompt_tokens': 664, 'completion_tokens': 162, 'total_tokens': 826})

In [22]:
run.status

'completed'

## 6단계: 어시스턴트의 응답 표시
실행이 완료되면 어시스턴트가 스레드에 추가한 메시지를 나열할 수 있습니다.

In [41]:
messages = client.beta.threads.messages.list(
  thread_id=thread.id
)

In [77]:
thread_messages = []

for i in range(len(messages.data)-1, -1, -1):
    role = messages.data[i].role
    message = vars(messages.data[i].content[0])['text'].value
    print(role, ":")
    print(message)

user :
I need to solve the equation `3x + 11 = 14`. Can you help me?
assistant :
Yes, I can help you solve the equation \(3x + 11 = 14\). To do this, we'll follow these steps:

1. Subtract 11 from both sides of the equation to isolate the term with \(x\) on one side.
2. Divide both sides by 3 to solve for \(x\).

I'll calculate the solution now.
assistant :
The solution to the equation \(3x + 11 = 14\) is \(x = 1\).
